In [3]:
!pip install qiskit qiskit-aer matplotlib pylatexenc

Problem 3

In [5]:
from qiskit import QuantumCircuit
from qiskit.quantum_info import Statevector
from qiskit.visualization import plot_bloch_multivector
import matplotlib.pyplot as plt

# Create a 4-qubit quantum circuit
qc = QuantumCircuit(4)

# Initialize to |1011>
# In Qiskit ordering: |q3 q2 q1 q0> = |1 0 1 1|
qc.x(0)   # q0 = 1
qc.x(1)   # q1 = 1
# q2 stays 0
qc.x(3)   # q3 = 1

# Draw the circuit
print(qc.draw())

# Get the statevector
state = Statevector.from_instruction(qc)

# Print the statevector
print("Statevector:")
print(state)

# Plot Bloch sphere for all qubits
plot_bloch_multivector(state)
plt.show()

     ┌───┐
q_0: ┤ X ├
     ├───┤
q_1: ┤ X ├
     └───┘
q_2: ─────
     ┌───┐
q_3: ┤ X ├
     └───┘
Statevector:
Statevector([0.+0.j, 0.+0.j, 0.+0.j, 0.+0.j, 0.+0.j, 0.+0.j, 0.+0.j,
             0.+0.j, 0.+0.j, 0.+0.j, 0.+0.j, 1.+0.j, 0.+0.j, 0.+0.j,
             0.+0.j, 0.+0.j],
            dims=(2, 2, 2, 2))


In [6]:
fig = plot_bloch_multivector(state)
fig.savefig("bloch_sphere_1011.png")
plt.show()

Problem 4

In [22]:
# ============================================================
# Install first (run once):
# pip install qiskit qiskit-aer matplotlib pylatexenc
# ============================================================

from qiskit import QuantumCircuit, QuantumRegister, ClassicalRegister
from qiskit_aer import AerSimulator
from qiskit.visualization import plot_histogram
import matplotlib.pyplot as plt

# Step 1: Create registers
q = QuantumRegister(3, 'q')   # 3 qubits: q0, q1, q2
c = ClassicalRegister(3, 'C') # 3 classical bits
qc = QuantumCircuit(q, c)

# Step 2: Apply gates (exactly as shown in the image)
qc.h(q[2])          # Hadamard on q2  → puts q2 into superposition (|0⟩+|1⟩)/√2

qc.cx(q[2], q[1])   # CNOT: q2 controls q1 (first blue dot → X on q1)

qc.cx(q[2], q[0])   # CNOT: q2 controls q0 (second blue dot → first X on q0)
qc.cx(q[2], q[0])   # CNOT again: q2 controls q0 (third blue dot → second X on q0)

# Step 3: Measure all qubits into classical register
qc.measure(q[0], c[0])
qc.measure(q[1], c[1])
qc.measure(q[2], c[2])

# Step 4: Draw the circuit
print(qc.draw(output='text'))

# Step 5: Run simulation
simulator = AerSimulator()
job = simulator.run(qc, shots=1024)
result = job.result()
counts = result.get_counts()

print("\nMeasurement results:", counts)

# Step 6: Plot histogram
fig = plot_histogram(counts)
fig.suptitle("Histogram of Measurement Outcomes", fontsize=13)
plt.savefig("histogram.png", dpi=150, bbox_inches='tight')
plt.show()

               ┌───┐   ┌───┐┌─┐   
q_0: ──────────┤ X ├───┤ X ├┤M├───
          ┌───┐└─┬─┘┌─┐└─┬─┘└╥┘   
q_1: ─────┤ X ├──┼──┤M├──┼───╫────
     ┌───┐└─┬─┘  │  └╥┘  │   ║ ┌─┐
q_2: ┤ H ├──■────■───╫───■───╫─┤M├
     └───┘           ║       ║ └╥┘
C: 3/════════════════╩═══════╩══╩═
                     1       0  2 

Measurement results: {'110': 518, '000': 506}


<Figure size 640x480 with 0 Axes>

In [27]:
fig = plot_histogram(counts)
fig.suptitle("Histogram of Measurement Outcomes", fontsize=13)

# Fix overlap
fig.tight_layout(rect=[0, 0, 1, 0.92])

fig.savefig("histogram.png", dpi=150, bbox_inches='tight')
plt.show()

Problem 5


In [25]:
# ============================================================
# INSTALL FIRST (run once):
# pip install qiskit qiskit-aer matplotlib pylatexenc
# ============================================================

from qiskit import QuantumCircuit, QuantumRegister, ClassicalRegister
from qiskit_aer import AerSimulator
from qiskit.visualization import plot_histogram
import matplotlib.pyplot as plt

# ── Step 1: Build the circuit ────────────────────────────────
q = QuantumRegister(2, 'q')    # 2 qubits
c = ClassicalRegister(2, 'c')  # 2 classical bits for measurement
qc = QuantumCircuit(q, c)

# q[0] = bottom wire, q[1] = top wire  (Qiskit numbers bottom-up)

qc.h(q[0])            # H on bottom qubit (control preparation)
qc.cx(q[0], q[1])     # CNOT: q[0] controls q[1]  (the ⊕ symbol)
qc.s(q[1])            # S gate on top qubit
qc.h(q[1])            # H gate on top qubit
qc.t(q[1])            # T gate on top qubit
qc.h(q[1])            # H gate on top qubit

# ── Step 2: Add measurements ─────────────────────────────────
qc.measure(q[0], c[0])
qc.measure(q[1], c[1])

# ── Step 3: Draw the circuit ─────────────────────────────────
print("=" * 55)
print("  Circuit Diagram")
print("=" * 55)
print(qc.draw(output='text'))

# Save circuit as image
circuit_fig = qc.draw(output='mpl', style='iqp')
circuit_fig.suptitle("Quantum Circuit (Question 5)", fontsize=13)
circuit_fig.savefig("circuit_q5.png", dpi=150, bbox_inches='tight')
print("Circuit saved as circuit_q5.png")

# ── Step 4: Simulate ─────────────────────────────────────────
simulator = AerSimulator()
job = simulator.run(qc, shots=1024)
result = job.result()
sim_counts = result.get_counts()

print("\nSimulator results:", sim_counts)

# ── Step 5: Plot simulator histogram ─────────────────────────
fig_sim = plot_histogram(sim_counts, title="Simulator Results (1024 shots)",
                         color='steelblue')
fig_sim.savefig("histogram_simulator.png", dpi=150, bbox_inches='tight')
plt.show()
print("Histogram saved as histogram_simulator.png")

  Circuit Diagram
     ┌───┐          ┌─┐                  
q_0: ┤ H ├──■───────┤M├──────────────────
     └───┘┌─┴─┐┌───┐└╥┘┌───┐┌───┐┌───┐┌─┐
q_1: ─────┤ X ├┤ S ├─╫─┤ H ├┤ T ├┤ H ├┤M├
          └───┘└───┘ ║ └───┘└───┘└───┘└╥┘
c: 2/════════════════╩═════════════════╩═
                     0                 1 
Circuit saved as circuit_q5.png

Simulator results: {'11': 446, '01': 85, '00': 405, '10': 88}
Histogram saved as histogram_simulator.png


In [28]:
pip install qiskit-ibm-runtime

In [ ]:
# ============================================================
# INSTALL FIRST (run once):
# pip install qiskit qiskit-ibm-runtime matplotlib pylatexenc
# ============================================================

from qiskit import QuantumCircuit, QuantumRegister, ClassicalRegister
from qiskit.transpiler import generate_preset_pass_manager
from qiskit.visualization import plot_histogram
from qiskit_ibm_runtime import QiskitRuntimeService, SamplerV2 as Sampler
import matplotlib.pyplot as plt

# ============================================================
# STEP 0: CONNECT TO IBM QUANTUM
# ============================================================
# # Option A: if you already saved your credentials earlier
# service = QiskitRuntimeService()

# Option B: use token directly every time
service = QiskitRuntimeService(
    channel="ibm_cloud",
    token="your_token",
    instance="your_instance"   # recommended if you have it
)

# Pick a real backend (not a simulator)
backend = service.least_busy(
    operational=True,
    simulator=False,
    min_num_qubits=2
)

print("Using backend:", backend.name)

# ============================================================
# STEP 1: BUILD THE SAME CIRCUIT
# ============================================================
q = QuantumRegister(2, 'q')
c = ClassicalRegister(2, 'c')
qc = QuantumCircuit(q, c)

# q[0] = bottom wire, q[1] = top wire
qc.h(q[0])
qc.cx(q[0], q[1])
qc.s(q[1])
qc.h(q[1])
qc.t(q[1])
qc.h(q[1])

qc.measure(q[0], c[0])
qc.measure(q[1], c[1])

print("=" * 55)
print("Original Circuit")
print("=" * 55)
print(qc.draw(output='text'))

# Save original circuit image
circuit_fig = qc.draw(output='mpl', style='iqp')
circuit_fig.suptitle("Quantum Circuit (Question 5)", fontsize=13)
circuit_fig.savefig("circuit_q5.png", dpi=150, bbox_inches='tight')
print("Circuit saved as circuit_q5.png")

# ============================================================
# STEP 2: TRANSPLE FOR THE REAL HARDWARE
# ============================================================
pm = generate_preset_pass_manager(backend=backend, optimization_level=1)
isa_circuit = pm.run(qc)

print("=" * 55)
print("Transpiled ISA Circuit")
print("=" * 55)
print(isa_circuit.draw(output='text'))

# Save transpiled circuit image
isa_fig = isa_circuit.draw(output='mpl', style='iqp')
isa_fig.suptitle(f"Transpiled Circuit for {backend.name}", fontsize=13)
isa_fig.savefig("transpiled_q5.png", dpi=150, bbox_inches='tight')
print("Transpiled circuit saved as transpiled_q5.png")

# ============================================================
# STEP 3: RUN ON REAL IBM HARDWARE
# ============================================================
sampler = Sampler(mode=backend)

# Run one PUB containing the transpiled circuit
job = sampler.run([(isa_circuit,)], shots=1024)

print("Job ID:", job.job_id())
print("Job submitted. Waiting for result...")

result = job.result()

# Extract measured bitstrings from the default classical register data
pub_result = result[0]
bitstrings = pub_result.data.c.get_bitstrings()

# Convert bitstrings to counts dictionary
hardware_counts = {}
for b in bitstrings:
    hardware_counts[b] = hardware_counts.get(b, 0) + 1

print("\nHardware results:", hardware_counts)

# ============================================================
# STEP 4: PLOT HARDWARE HISTOGRAM
# ============================================================
fig_hw = plot_histogram(hardware_counts)
fig_hw.suptitle(f"IBM Hardware Results ({backend.name}, 1024 shots)", fontsize=13)
fig_hw.tight_layout(rect=[0, 0, 1, 0.94])
fig_hw.savefig("histogram_hardware.png", dpi=150, bbox_inches='tight')
plt.show()

print("Hardware histogram saved as histogram_hardware.png")

Using backend: ibm_marrakesh
Original Circuit
     ┌───┐          ┌─┐                  
q_0: ┤ H ├──■───────┤M├──────────────────
     └───┘┌─┴─┐┌───┐└╥┘┌───┐┌───┐┌───┐┌─┐
q_1: ─────┤ X ├┤ S ├─╫─┤ H ├┤ T ├┤ H ├┤M├
          └───┘└───┘ ║ └───┘└───┘└───┘└╥┘
c: 2/════════════════╩═════════════════╩═
                     0                 1 
Circuit saved as circuit_q5.png
Transpiled ISA Circuit
global phase: 7π/8
         ┌─────────┐┌────┐┌─────────┐         ┌─┐                             »
q_0 -> 0 ┤ Rz(π/2) ├┤ √X ├┤ Rz(π/2) ├─■───────┤M├─────────────────────────────»
         ├─────────┤├────┤├─────────┤ │ ┌────┐└╥┘┌──────────┐┌────┐┌─────────┐»
q_1 -> 1 ┤ Rz(π/2) ├┤ √X ├┤ Rz(π/2) ├─■─┤ √X ├─╫─┤ Rz(3π/4) ├┤ √X ├┤ Rz(π/2) ├»
         └─────────┘└────┘└─────────┘   └────┘ ║ └──────────┘└────┘└─────────┘»
    c: 2/══════════════════════════════════════╩══════════════════════════════»
                                               0                              »
«            
«q_0 -> 0 ──